> **Group Comparison - Optimized Models**
<br>` Comparing best models from 3 students `
<br>` Student 1: Ensemble (BaggingClassifier) `
<br>` Student 2: Non-Linear (DecisionTree) `
<br>` Student 3: Support Vector (RidgeClassifier) `

**Import the main libraries**

In [1]:
import numpy as np
import pandas as pd

from time import time

import os
data_path = '../data'

_import the local library_

In [2]:
# add parent folder path where lib folder is
import sys
if ".." not in sys.path:import sys; sys.path.insert(0, '..') 

In [3]:
from mylib import show_labels_dist, show_metrics, bias_var_metrics

/usr/local/lib/python3.12/dist-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


**Import the Dataset**

In [4]:
# Using boosted Train and preprocessed Test

data_file = os.path.join(data_path, 'NSL_boosted-2.csv') 
train_df = pd.read_csv(data_file)
print('Train Dataset: {} rows, {} columns'.format(train_df.shape[0], train_df.shape[1]))

data_file = os.path.join(data_path, 'NSL_ppTest.csv') 
test_df = pd.read_csv(data_file)
print('Test Dataset: {} rows, {} columns'.format(test_df.shape[0], test_df.shape[1]))

Train Dataset: 63280 rows, 43 columns
Test Dataset: 22544 rows, 43 columns


***
**Data Preparation** (same as individual notebooks)

In [5]:
# Identify columns with only one value 
n_eq_one = []

for col in train_df.columns:
    lctrn = len(train_df[col].unique())
    lctst = len(test_df[col].unique())
    if (lctrn == 1) and (lctrn == lctst): 
        n_eq_one.append(train_df[col].name)

# Drop columns with only one value
if len(n_eq_one) > 0:
    print('Dropping single-valued features:', n_eq_one)
    train_df.drop(n_eq_one, axis=1, inplace=True)
    test_df.drop(n_eq_one, axis=1, inplace=True)

Dropping single-valued features: ['num_outbound_cmds']


In [6]:
# Combine for processing
combined_df = pd.concat([train_df, test_df])
print('Combined Dataset: {} rows, {} columns'.format(
    combined_df.shape[0], combined_df.shape[1]))

Combined Dataset: 85824 rows, 42 columns


**Classification Target** - Two-class (normal vs attack)

In [7]:
# Two-class: Reduce the detailed attack labels to 'normal' or 'attack'
labels_df = combined_df['label'].copy()
labels_df[labels_df != 'normal'] = 'attack'

# drop target features 
combined_df.drop(['label'], axis=1, inplace=True)
combined_df.drop(['atakcat'], axis=1, inplace=True)

**One-Hot Encoding** the categorical features

In [8]:
# put the names into a python list - for pandas.get_dummies()
categori = combined_df.select_dtypes(include=['object']).columns
category_cols = categori.tolist()
print('Categorical columns:', category_cols)

# Apply one-hot encoding
features_df = pd.get_dummies(combined_df, columns=category_cols)
print('Features after encoding:', features_df.shape[1], 'columns')

Categorical columns: ['protocol_type', 'service', 'flag']
Features after encoding: 121 columns


In [9]:
# generate a list of numeric columns for scaling
numeri = combined_df.select_dtypes(include=['float64','int64']).columns
print('Numeric columns:', len(numeri.to_list()))

Numeric columns: 37


***
**Create Test // Train Datasets**

In [10]:
# Restore the train // test split
features_train = features_df.iloc[:len(train_df),:].copy()
features_train.reset_index(inplace=True, drop=True)

features_test = features_df.iloc[len(train_df):,:].copy()
features_test.reset_index(inplace=True, drop=True)

labels_train = labels_df[:len(train_df)]
labels_train.reset_index(inplace=True, drop=True)

labels_test = labels_df[len(train_df):]
labels_test.reset_index(inplace=True, drop=True)

**Scaling** comes _after_ test // train split

In [11]:
from sklearn.preprocessing import MinMaxScaler

for i in numeri:
    arr = np.array(features_train[i])
    scale = MinMaxScaler().fit(arr.reshape(-1, 1))
    features_train[i] = scale.transform(arr.reshape(len(arr),1))

    arr = np.array(features_test[i])
    features_test[i] = scale.transform(arr.reshape(len(arr),1))

print('Scaling complete')

Scaling complete


In [12]:
# dataset names (compatibility)
X_train = features_train
y_train = labels_train
X_test = features_test
y_test = labels_test

**Target Label Distributions**

In [13]:
# from our local library
show_labels_dist(X_train, X_test, y_train, y_test)

features_train: 63280 rows, 121 columns
features_test:  22544 rows, 121 columns

labels_train: 63280 rows, 1 column
labels_test:  22544 rows, 1 column

Frequency and Distribution of labels
        label  %_train  label  %_test
label                                
normal  33672    53.21   9711   43.08
attack  29608    46.79  12833   56.92


***
***
## GROUP COMPARISON - OPTIMIZED MODELS
***

**Define the 3 Best Optimized Models**
<br>Each from a different classifier category

In [14]:
# Classifier imports
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import RidgeClassifier

# Define the 3 best optimized models from each student
# Using n_jobs=1 to avoid CPU overload
models = [
    # Student 1: Ensemble - BaggingClassifier (Best: Bagging_150est)
    ("Ensemble_Bagging", BaggingClassifier(
        n_estimators=150,
        max_samples=0.85,
        random_state=42,
        n_jobs=1  # Single CPU to avoid overload
    )),

    # Student 2: Non-Linear - DecisionTree (Best: DT_depth20)
    ("NonLinear_DecisionTree", DecisionTreeClassifier(
        max_depth=20,
        random_state=42
    )),

    # Student 3: Support Vector - RidgeClassifier (Best: Ridge_balanced)
    ("SupportVector_Ridge", RidgeClassifier(
        class_weight='balanced',
        random_state=42
    )),
]

print('Models for comparison:')
for name, clf in models:
    print(f'  - {name}')

Models for comparison:
  - Ensemble_Bagging
  - NonLinear_DecisionTree
  - SupportVector_Ridge


***
### COMPARISON 1: Accuracy and MCC Metrics
***

In [15]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.metrics import f1_score, matthews_corrcoef
from sklearn.metrics import precision_score, recall_score

# Store results
comparison_results = {}
predictions = {}

print('='*70)
print('COMPARISON 1: ACCURACY AND MCC METRICS')
print('='*70)

for name, clf in models:
    trs = time()
    print(f'\nTraining: {name}')
    
    # Fit and predict
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    predictions[name] = y_pred
    
    tre = time() - trs
    
    # Calculate metrics
    comparison_results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred, average='weighted'),
        'precision': precision_score(y_test, y_pred, average='weighted'),
        'recall': recall_score(y_test, y_pred, average='weighted'),
        'mcc': matthews_corrcoef(y_test, y_pred),
        'time': tre
    }
    
    r = comparison_results[name]
    print(f'  Accuracy:          {r["accuracy"]:.4f}')
    print(f'  Balanced Accuracy: {r["balanced_accuracy"]:.4f}')
    print(f'  F1-Score:          {r["f1_score"]:.4f}')
    print(f'  MCC:               {r["mcc"]:.4f}')
    print(f'  Time:              {r["time"]:.2f}s')

COMPARISON 1: ACCURACY AND MCC METRICS

Training: Ensemble_Bagging
  Accuracy:          0.9100
  Balanced Accuracy: 0.9160
  F1-Score:          0.9104
  MCC:               0.8244
  Time:              79.41s

Training: NonLinear_DecisionTree
  Accuracy:          0.9057
  Balanced Accuracy: 0.9119
  F1-Score:          0.9061
  MCC:               0.8160
  Time:              0.87s

Training: SupportVector_Ridge
  Accuracy:          0.7727
  Balanced Accuracy: 0.7918
  F1-Score:          0.7716
  MCC:               0.5894
  Time:              0.38s


**Accuracy Comparison Table**

In [16]:
# Create comparison DataFrame
accuracy_df = pd.DataFrame({
    'Model': list(comparison_results.keys()),
    'Accuracy': [r['accuracy'] for r in comparison_results.values()],
    'Balanced_Accuracy': [r['balanced_accuracy'] for r in comparison_results.values()],
    'F1_Score': [r['f1_score'] for r in comparison_results.values()],
    'Precision': [r['precision'] for r in comparison_results.values()],
    'Recall': [r['recall'] for r in comparison_results.values()],
    'MCC': [r['mcc'] for r in comparison_results.values()],
})

# Sort by balanced accuracy
accuracy_df = accuracy_df.sort_values('Balanced_Accuracy', ascending=False)
accuracy_df = accuracy_df.reset_index(drop=True)

print('\n' + '='*70)
print('TABLE 1: ACCURACY COMPARISON')
print('='*70)
print(accuracy_df.to_string(index=False))


TABLE 1: ACCURACY COMPARISON
                 Model  Accuracy  Balanced_Accuracy  F1_Score  Precision   Recall      MCC
      Ensemble_Bagging  0.909954           0.916047  0.910372   0.916420 0.909954 0.824382
NonLinear_DecisionTree  0.905651           0.911854  0.906092   0.912417 0.905651 0.816016
   SupportVector_Ridge  0.772667           0.791802  0.771583   0.815230 0.772667 0.589356


**Detailed Classification Report for Each Model**

In [17]:
print('='*70)
print('DETAILED CLASSIFICATION REPORTS')
print('='*70)

for name, clf in models:
    print(f'\n--- {name} ---')
    show_metrics(y_test, predictions[name], clf.classes_)

DETAILED CLASSIFICATION REPORTS

--- Ensemble_Bagging ---
              pred:attack  pred:normal
train:attack        11191         1642
train:normal          388         9323

~~~~
   macro avg :  FPR = 0.084   FNR = 0.084
weighted avg :  FPR = 0.090   FNR = 0.090

~~~~
              precision    recall  f1-score   support

      attack      0.966     0.872     0.917     12833
      normal      0.850     0.960     0.902      9711

    accuracy                          0.910     22544
   macro avg      0.908     0.916     0.909     22544
weighted avg      0.916     0.910     0.910     22544

~~~~
MCC: Overall :  0.824

--- NonLinear_DecisionTree ---
              pred:attack  pred:normal
train:attack        11127         1706
train:normal          421         9290

~~~~
   macro avg :  FPR = 0.088   FNR = 0.088
weighted avg :  FPR = 0.094   FNR = 0.094

~~~~
              precision    recall  f1-score   support

      attack      0.964     0.867     0.913     12833
      normal      0.8

***
### COMPARISON 2: Bias-Variance Decomposition
***

In [ ]:
from mlxtend.evaluate import bias_variance_decomp
from sklearn.preprocessing import LabelEncoder

# bias_variance_decomp requires numeric targets
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# Store bias-variance results
bias_var_results = {}

print('='*70)
print('COMPARISON 2: BIAS-VARIANCE DECOMPOSITION')
print('='*70)

# Number of bootstrap rounds (reduced to 10 for CPU efficiency)
num_rounds = 10

for name, clf in models:
    trs = time()
    print(f'\nComputing Bias-Variance: {name}')

    # Need to recreate the model for bias_variance_decomp
    # Using n_jobs=1 to avoid CPU overload
    if 'Bagging' in name:
        clf_fresh = BaggingClassifier(n_estimators=150, max_samples=0.85, random_state=42, n_jobs=1)
    elif 'DecisionTree' in name:
        clf_fresh = DecisionTreeClassifier(max_depth=20, random_state=42)
    else:
        clf_fresh = RidgeClassifier(class_weight='balanced', random_state=42)

    avg_expected_loss, avg_bias, avg_var = bias_variance_decomp(
        clf_fresh,
        X_train.values, y_train_enc,
        X_test.values, y_test_enc,
        loss='0-1_loss',
        num_rounds=num_rounds,
        random_seed=42
    )

    tre = time() - trs

    bias_var_results[name] = {
        'bias': avg_bias,
        'variance': avg_var,
        'expected_loss': avg_expected_loss,
        'goodness': 1 - avg_expected_loss
    }

    print(f'  Average Bias:          {avg_bias:.4f}')
    print(f'  Average Variance:      {avg_var:.4f}')
    print(f'  Average Expected Loss: {avg_expected_loss:.4f}')
    print(f'  "Goodness":            {1-avg_expected_loss:.4f}')
    print(f'  Time:                  {tre:.2f}s')

COMPARISON 2: BIAS-VARIANCE DECOMPOSITION

Computing Bias-Variance: Ensemble_Bagging


**Bias-Variance Comparison Table**

In [ ]:
# Create bias-variance DataFrame
bias_var_df = pd.DataFrame({
    'Model': list(bias_var_results.keys()),
    'Bias': [r['bias'] for r in bias_var_results.values()],
    'Variance': [r['variance'] for r in bias_var_results.values()],
    'Expected_Loss': [r['expected_loss'] for r in bias_var_results.values()],
    'Goodness': [r['goodness'] for r in bias_var_results.values()],
})

# Sort by expected loss (lower is better)
bias_var_df = bias_var_df.sort_values('Expected_Loss', ascending=True)
bias_var_df = bias_var_df.reset_index(drop=True)

print('\n' + '='*70)
print('TABLE 2: BIAS-VARIANCE COMPARISON')
print('='*70)
print(bias_var_df.to_string(index=False))

***
### COMPARISON 3: Cross-Validation
***

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Cross-validation settings
cv_folds = 5
skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)

# Store CV results
cv_results = {}

print('='*70)
print(f'COMPARISON 3: {cv_folds}-FOLD CROSS-VALIDATION')
print('='*70)

for name, clf in models:
    trs = time()
    print(f'\nCross-Validating: {name}')

    # Need to recreate the model for cross-validation
    # Using n_jobs=1 to avoid CPU overload
    if 'Bagging' in name:
        clf_fresh = BaggingClassifier(n_estimators=150, max_samples=0.85, random_state=42, n_jobs=1)
    elif 'DecisionTree' in name:
        clf_fresh = DecisionTreeClassifier(max_depth=20, random_state=42)
    else:
        clf_fresh = RidgeClassifier(class_weight='balanced', random_state=42)

    # Cross-validation scores (n_jobs=1 for CV as well)
    cv_accuracy = cross_val_score(clf_fresh, X_train, y_train, cv=skf, scoring='accuracy', n_jobs=1)
    cv_balanced = cross_val_score(clf_fresh, X_train, y_train, cv=skf, scoring='balanced_accuracy', n_jobs=1)
    cv_f1 = cross_val_score(clf_fresh, X_train, y_train, cv=skf, scoring='f1_weighted', n_jobs=1)

    tre = time() - trs

    cv_results[name] = {
        'cv_accuracy_mean': cv_accuracy.mean(),
        'cv_accuracy_std': cv_accuracy.std(),
        'cv_balanced_mean': cv_balanced.mean(),
        'cv_balanced_std': cv_balanced.std(),
        'cv_f1_mean': cv_f1.mean(),
        'cv_f1_std': cv_f1.std(),
    }

    print(f'  CV Accuracy:          {cv_accuracy.mean():.4f} (+/- {cv_accuracy.std():.4f})')
    print(f'  CV Balanced Accuracy: {cv_balanced.mean():.4f} (+/- {cv_balanced.std():.4f})')
    print(f'  CV F1-Score:          {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})')
    print(f'  Time:                 {tre:.2f}s')

**Cross-Validation Comparison Table**

In [ ]:
# Create CV DataFrame
cv_df = pd.DataFrame({
    'Model': list(cv_results.keys()),
    'CV_Accuracy': [f"{r['cv_accuracy_mean']:.4f} (+/-{r['cv_accuracy_std']:.4f})" for r in cv_results.values()],
    'CV_Balanced_Acc': [f"{r['cv_balanced_mean']:.4f} (+/-{r['cv_balanced_std']:.4f})" for r in cv_results.values()],
    'CV_F1_Score': [f"{r['cv_f1_mean']:.4f} (+/-{r['cv_f1_std']:.4f})" for r in cv_results.values()],
})

print('\n' + '='*70)
print('TABLE 3: CROSS-VALIDATION COMPARISON')
print('='*70)
print(cv_df.to_string(index=False))

***
### COMPARISON 4: Per-Class Performance (Critical for IDS)
***
For Intrusion Detection Systems, per-class metrics are critical:
- **Attack Detection Rate (True Positive Rate)**: How many attacks are correctly identified
- **False Alarm Rate (False Positive Rate)**: How many normal connections are wrongly flagged
- **Attack Precision**: Of all predicted attacks, how many are actual attacks

In [ ]:
from sklearn.metrics import confusion_matrix

print('='*70)
print('COMPARISON 4: PER-CLASS PERFORMANCE (CRITICAL FOR IDS)')
print('='*70)

per_class_data = []
model_names = list(comparison_results.keys())

for name in model_names:
    cm = confusion_matrix(y_test, predictions[name], labels=['attack', 'normal'])
    # cm[0,0]=TP(attack), cm[0,1]=FN, cm[1,0]=FP, cm[1,1]=TN
    TP = cm[0, 0]  # True Positive (attack correctly detected)
    FN = cm[0, 1]  # False Negative (attack missed)
    FP = cm[1, 0]  # False Positive (normal flagged as attack)
    TN = cm[1, 1]  # True Negative (normal correctly identified)
    
    # Detection Rate (Recall for attack class) - most important for IDS
    detection_rate = TP / (TP + FN) if (TP + FN) > 0 else 0
    # False Alarm Rate (FPR)
    false_alarm_rate = FP / (FP + TN) if (FP + TN) > 0 else 0
    # Precision for attack class
    attack_precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    
    per_class_data.append({
        'Model': name,
        'Attack_Detection_Rate': detection_rate,
        'False_Alarm_Rate': false_alarm_rate,
        'Attack_Precision': attack_precision,
        'Attacks_Detected': TP,
        'Attacks_Missed': FN,
        'False_Alarms': FP
    })
    
    print(f'\n{name}:')
    print(f'  Attack Detection Rate: {detection_rate:.4f} ({TP}/{TP+FN} attacks detected)')
    print(f'  False Alarm Rate:      {false_alarm_rate:.4f} ({FP}/{FP+TN} false alarms)')
    print(f'  Attack Precision:      {attack_precision:.4f}')

per_class_df = pd.DataFrame(per_class_data).round(4)
print('\n' + '='*70)
print('TABLE 4: PER-CLASS PERFORMANCE')
print('='*70)
print(per_class_df[['Model', 'Attack_Detection_Rate', 'False_Alarm_Rate', 'Attack_Precision']].to_string(index=False))

***
### COMPARISON 5: Generalization Analysis
***
Comparing Train (CV) vs Test performance to assess overfitting.
- A large gap indicates the model doesn't generalize well to unseen data
- NSL-KDD test set contains novel attack types not in training, making this analysis critical

In [ ]:
print('='*70)
print('COMPARISON 5: GENERALIZATION ANALYSIS (Train CV vs Test)')
print('='*70)

gen_data = []
for name in model_names:
    train_cv = cv_results[name]['cv_balanced_mean']
    test_score = comparison_results[name]['balanced_accuracy']
    gap = train_cv - test_score
    gap_pct = (gap / train_cv * 100) if train_cv > 0 else 0
    
    gen_data.append({
        'Model': name,
        'Train_CV': train_cv,
        'Test': test_score,
        'Gap': gap,
        'Gap_%': gap_pct,
    })
    
    print(f'\n{name}:')
    print(f'  Train CV Balanced Acc: {train_cv:.4f}')
    print(f'  Test Balanced Acc:     {test_score:.4f}')
    print(f'  Performance Gap:       {gap:.4f} ({gap_pct:.1f}% drop)')

gen_df = pd.DataFrame(gen_data).round(4)
print('\n' + '='*70)
print('TABLE 5: GENERALIZATION ANALYSIS')
print('='*70)
print(gen_df.to_string(index=False))

print('\nNote: The NSL-KDD test set intentionally contains novel attack types')
print('not present in training data, which explains the performance gap.')

***
### VISUALIZATIONS
***

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

**Visualization 1: Balanced Accuracy Comparison**

In [ ]:
# Bar chart - Balanced Accuracy
fig, ax = plt.subplots(figsize=(10, 6))

model_names = list(comparison_results.keys())
bal_acc = [comparison_results[m]['balanced_accuracy'] for m in model_names]

colors = ['#2ecc71', '#3498db', '#e74c3c']  # Green, Blue, Red
bars = ax.bar(model_names, bal_acc, color=colors, edgecolor='black', linewidth=1.2)

# Add value labels on bars
for bar, val in zip(bars, bal_acc):
    ax.annotate(f'{val:.4f}', 
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 5), textcoords='offset points',
                ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Balanced Accuracy', fontsize=12)
ax.set_xlabel('Model', fontsize=12)
ax.set_title('Balanced Accuracy Comparison - Optimized Models', fontsize=14, fontweight='bold')
ax.set_ylim(0.7, 1.0)
ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('../figures/comparison_balanced_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

**Visualization 2: MCC Comparison**

In [ ]:
# Bar chart - MCC
fig, ax = plt.subplots(figsize=(10, 6))

mcc_vals = [comparison_results[m]['mcc'] for m in model_names]

bars = ax.bar(model_names, mcc_vals, color=colors, edgecolor='black', linewidth=1.2)

# Add value labels on bars
for bar, val in zip(bars, mcc_vals):
    ax.annotate(f'{val:.4f}', 
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 5), textcoords='offset points',
                ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Matthews Correlation Coefficient (MCC)', fontsize=12)
ax.set_xlabel('Model', fontsize=12)
ax.set_title('MCC Comparison - Optimized Models', fontsize=14, fontweight='bold')
ax.set_ylim(0.5, 1.0)
ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('../figures/comparison_mcc.png', dpi=150, bbox_inches='tight')
plt.show()

**Visualization 3: Bias-Variance Comparison**

In [ ]:
# Grouped bar chart - Bias and Variance
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(model_names))
width = 0.25

bias_vals = [bias_var_results[m]['bias'] for m in model_names]
var_vals = [bias_var_results[m]['variance'] for m in model_names]
loss_vals = [bias_var_results[m]['expected_loss'] for m in model_names]

bars1 = ax.bar(x - width, bias_vals, width, label='Bias', color='#3498db', edgecolor='black')
bars2 = ax.bar(x, var_vals, width, label='Variance', color='#f39c12', edgecolor='black')
bars3 = ax.bar(x + width, loss_vals, width, label='Expected Loss', color='#e74c3c', edgecolor='black')

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}',
                    xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Value', fontsize=12)
ax.set_xlabel('Model', fontsize=12)
ax.set_title('Bias-Variance Decomposition Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15)
ax.legend()
ax.set_ylim(0, 0.35)

plt.tight_layout()
plt.savefig('../figures/comparison_bias_variance.png', dpi=150, bbox_inches='tight')
plt.show()

**Visualization 4: Cross-Validation Comparison**

In [ ]:
# Bar chart with error bars - CV Balanced Accuracy
fig, ax = plt.subplots(figsize=(10, 6))

cv_means = [cv_results[m]['cv_balanced_mean'] for m in model_names]
cv_stds = [cv_results[m]['cv_balanced_std'] for m in model_names]

bars = ax.bar(model_names, cv_means, yerr=cv_stds, capsize=5,
              color=colors, edgecolor='black', linewidth=1.2)

# Add value labels
for bar, val, std in zip(bars, cv_means, cv_stds):
    ax.annotate(f'{val:.4f}', 
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height() + std),
                xytext=(0, 5), textcoords='offset points',
                ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('CV Balanced Accuracy', fontsize=12)
ax.set_xlabel('Model', fontsize=12)
ax.set_title(f'{cv_folds}-Fold Cross-Validation Comparison', fontsize=14, fontweight='bold')
ax.set_ylim(0.7, 1.0)
ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('../figures/comparison_cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()

**Visualization 5: Confusion Matrices**

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (name, clf) in enumerate(models):
    cm = confusion_matrix(y_test, predictions[name])
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['attack', 'normal'], 
                yticklabels=['attack', 'normal'],
                annot_kws={'size': 14})
    
    axes[idx].set_title(f'{name}\nBal_Acc: {comparison_results[name]["balanced_accuracy"]:.4f}', 
                        fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.suptitle('Confusion Matrix Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/comparison_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

**Visualization 6: All Metrics Summary**

In [ ]:
# Radar/Spider chart style comparison
fig, ax = plt.subplots(figsize=(12, 6))

metrics = ['Accuracy', 'Bal_Accuracy', 'F1_Score', 'Precision', 'Recall', 'MCC']
x = np.arange(len(metrics))
width = 0.25

for idx, name in enumerate(model_names):
    values = [
        comparison_results[name]['accuracy'],
        comparison_results[name]['balanced_accuracy'],
        comparison_results[name]['f1_score'],
        comparison_results[name]['precision'],
        comparison_results[name]['recall'],
        comparison_results[name]['mcc']
    ]
    ax.bar(x + (idx - 1) * width, values, width, label=name, color=colors[idx], edgecolor='black')

ax.set_ylabel('Score', fontsize=12)
ax.set_xlabel('Metric', fontsize=12)
ax.set_title('All Metrics Comparison - Optimized Models', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend(loc='lower right')
ax.set_ylim(0.6, 1.0)

plt.tight_layout()
plt.savefig('../figures/comparison_all_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

***
### FINAL SUMMARY AND CONCLUSION
***

In [ ]:
print('='*70)
print('FINAL SUMMARY - GROUP COMPARISON')
print('='*70)

# Find best model by different criteria
best_by_bal_acc = max(comparison_results.keys(), key=lambda x: comparison_results[x]['balanced_accuracy'])
best_by_mcc = max(comparison_results.keys(), key=lambda x: comparison_results[x]['mcc'])
best_by_loss = min(bias_var_results.keys(), key=lambda x: bias_var_results[x]['expected_loss'])
best_by_cv = max(cv_results.keys(), key=lambda x: cv_results[x]['cv_balanced_mean'])

print('\n--- BEST MODEL BY CRITERION ---')
print(f'  Best by Balanced Accuracy: {best_by_bal_acc} ({comparison_results[best_by_bal_acc]["balanced_accuracy"]:.4f})')
print(f'  Best by MCC:               {best_by_mcc} ({comparison_results[best_by_mcc]["mcc"]:.4f})')
print(f'  Best by Expected Loss:     {best_by_loss} ({bias_var_results[best_by_loss]["expected_loss"]:.4f})')
print(f'  Best by CV Bal_Accuracy:   {best_by_cv} ({cv_results[best_by_cv]["cv_balanced_mean"]:.4f})')

print('\n--- SUMMARY TABLES ---')
print('\nTABLE 1: Accuracy Metrics')
print(accuracy_df.to_string(index=False))

print('\nTABLE 2: Bias-Variance')
print(bias_var_df.to_string(index=False))

print('\nTABLE 3: Cross-Validation')
print(cv_df.to_string(index=False))

In [ ]:
# Comprehensive Winner Analysis
print('\n' + '='*70)
print('COMPREHENSIVE ANALYSIS AND CONCLUSION')
print('='*70)

# Winner by each criterion
model_names = list(comparison_results.keys())
winners = {}

# Accuracy metrics
winners['Balanced_Accuracy'] = max(model_names, key=lambda x: comparison_results[x]['balanced_accuracy'])
winners['MCC'] = max(model_names, key=lambda x: comparison_results[x]['mcc'])

# Bias-Variance
winners['Expected_Loss'] = min(model_names, key=lambda x: bias_var_results[x]['expected_loss'])

# Cross-Validation
winners['CV_Performance'] = max(model_names, key=lambda x: cv_results[x]['cv_balanced_mean'])

# Per-class (from per_class_df)
winners['Attack_Detection'] = per_class_df.loc[per_class_df['Attack_Detection_Rate'].idxmax(), 'Model']
winners['Lowest_False_Alarm'] = per_class_df.loc[per_class_df['False_Alarm_Rate'].idxmin(), 'Model']

# Generalization
winners['Best_Generalization'] = gen_df.loc[gen_df['Gap'].idxmin(), 'Model']

print('\n--- WINNER BY CRITERION ---')
for criterion, winner in winners.items():
    print(f'  {criterion}: {winner}')

# Count wins
win_count = {name: 0 for name in model_names}
for winner in winners.values():
    win_count[winner] += 1

print('\n--- WIN COUNT ---')
for name, count in sorted(win_count.items(), key=lambda x: x[1], reverse=True):
    print(f'  {name}: {count}/7 criteria')

overall_best = max(win_count.items(), key=lambda x: x[1])[0]

print(f'''
{'='*70}
FINAL CONCLUSION
{'='*70}

OVERALL BEST MODEL: {overall_best}

DETAILED COMPARISON:

1. ENSEMBLE (BaggingClassifier) - Student 1:
   - Balanced Accuracy: {comparison_results['Ensemble_Bagging']['balanced_accuracy']:.4f}
   - MCC: {comparison_results['Ensemble_Bagging']['mcc']:.4f}
   - Attack Detection: {per_class_df[per_class_df['Model']=='Ensemble_Bagging']['Attack_Detection_Rate'].values[0]:.4f}
   - False Alarm Rate: {per_class_df[per_class_df['Model']=='Ensemble_Bagging']['False_Alarm_Rate'].values[0]:.4f}
   
2. NON-LINEAR (DecisionTree) - Student 2:
   - Balanced Accuracy: {comparison_results['NonLinear_DecisionTree']['balanced_accuracy']:.4f}
   - MCC: {comparison_results['NonLinear_DecisionTree']['mcc']:.4f}
   - Attack Detection: {per_class_df[per_class_df['Model']=='NonLinear_DecisionTree']['Attack_Detection_Rate'].values[0]:.4f}
   - False Alarm Rate: {per_class_df[per_class_df['Model']=='NonLinear_DecisionTree']['False_Alarm_Rate'].values[0]:.4f}

3. SUPPORT VECTOR (RidgeClassifier) - Student 3:
   - Balanced Accuracy: {comparison_results['SupportVector_Ridge']['balanced_accuracy']:.4f}
   - MCC: {comparison_results['SupportVector_Ridge']['mcc']:.4f}
   - Attack Detection: {per_class_df[per_class_df['Model']=='SupportVector_Ridge']['Attack_Detection_Rate'].values[0]:.4f}
   - False Alarm Rate: {per_class_df[per_class_df['Model']=='SupportVector_Ridge']['False_Alarm_Rate'].values[0]:.4f}

KEY INSIGHTS:

1. Ensemble methods (Bagging) provide the best overall performance
   with highest detection rate and lowest false alarms.

2. Decision Tree offers comparable performance to Bagging with
   significantly faster training time - good for real-time systems.

3. Ridge Classifier shows lowest performance, indicating that linear
   decision boundaries are insufficient for network intrusion detection.

4. All models show a performance gap between training and test data,
   which is expected as NSL-KDD test set contains novel attack types.

5. For IDS deployment, Bagging is recommended if computational resources
   allow; otherwise, Decision Tree is a practical alternative.
''')

***
### SAVE RESULTS
***

In [ ]:
import pickle

# Save all comparison results
all_comparison_data = {
    'accuracy_results': comparison_results,
    'bias_var_results': bias_var_results,
    'cv_results': cv_results,
    'per_class_results': per_class_df.to_dict(),
    'generalization_results': gen_df.to_dict(),
    'accuracy_df': accuracy_df,
    'bias_var_df': bias_var_df,
    'cv_df': cv_df,
    'per_class_df': per_class_df,
    'gen_df': gen_df,
    'best_model': overall_best,
    'winners': winners
}

with open('../data/group_comparison_results.pkl', 'wb') as f:
    pickle.dump(all_comparison_data, f)

print('Results saved to: ../data/group_comparison_results.pkl')

# Also save tables as CSV for easy import to Word
accuracy_df.to_csv('../data/comparison_accuracy_table.csv', index=False)
bias_var_df.to_csv('../data/comparison_biasvar_table.csv', index=False)
cv_df.to_csv('../data/comparison_cv_table.csv', index=False)
per_class_df.to_csv('../data/comparison_perclass_table.csv', index=False)
gen_df.to_csv('../data/comparison_generalization_table.csv', index=False)

print('Tables saved as CSV files:')
print('  - comparison_accuracy_table.csv')
print('  - comparison_biasvar_table.csv')
print('  - comparison_cv_table.csv')
print('  - comparison_perclass_table.csv')
print('  - comparison_generalization_table.csv')

In [ ]:
print('\n' + '='*70)
print('FIGURES SAVED')
print('='*70)
print('The following figures have been saved to ../figures/')
print('  1. comparison_balanced_accuracy.png')
print('  2. comparison_mcc.png')
print('  3. comparison_bias_variance.png')
print('  4. comparison_cross_validation.png')
print('  5. comparison_confusion_matrices.png')
print('  6. comparison_all_metrics.png')
print('\nThese figures are for reference. For the report,')
print('create simpler charts in Word/Excel as per requirements.')